**D. Psychosocial Features:**                   
22. **partner_support_score** - Score 0-10       
   - Normal(6.5, 2.5), lower in joint families   
    
23. **family_support_score** - Score 0-10
    - Higher in joint families
    
24. **domestic_violence_exposure** - {Never, Rarely, Sometimes, Often}
    - Never: 60%, Rarely: 20%, Sometimes: 15%, Often: 5%
    
25. **social_isolation_score** - Score 0-10
    - Higher in nuclear families, urban areas
    
26. **sleep_quality** - Score 0-10
    - Affected by baby health, support
    
27. **stressful_life_events** - Count (0-5)
    - Examples: death in family, job loss, migration
    
28. **mental_health_awareness** - {Low, Medium, High}
    - Correlated with education

----
----
----

#### 22. **partner_support_score:** (0-10)
- Perceived emotional/practical support from spouse/partner.

- **Statistics:** NDHS/WHO surveys show moderate-to-good support common, but lower in joint/extended families due to distribution of support.

- **Distribution:** Mean ~6.5, SD ~2.5. Slightly lower in joint families (mean ~6.0), higher in nuclear (mean ~7.0).

#### 23. **family_support_score:** (0-10)

- Perceived practical/emotional support from entire family (including elders).

- **Statistics:** Joint families report higher support (mean ~7.7, SD ~1.6), nuclear families lower (mean ~5.5, SD ~2.1).

#### 24. **domestic_violence_exposure:** {Never, Rarely, Sometimes, Often}
- **Statistics (NDHS 2022):**  
    - Never: 60%  
    - Rarely: 20%  
    - Sometimes: 15%  
    - Often: 5%

- About 23% of women have ever experienced physical violence, 13% emotional, and 7% sexual (lifetime), but "Often" is a minority.

- Increased risk among: less educated, poorer, Dalit/Madhesi, rural.

#### 25. **social_isolation_score:** (0-10)

- **Definition:** Sense of loneliness or social disconnectedness.

- **Statistics:** Higher in nuclear families and urban areas (mean ~4.2 nuclear/urban, ~2.6 joint/rural).

- Correlates strongly with family structure and to some extent residence.

#### 26. **sleep_quality:** (0-10, higher = better)

- **Statistics:** Median ~6.6, SD~2.1 (postnatal); 28% of mothers report poor quality.

- Lower with poor baby health, low support, high isolation, violence exposure.

#### 27. **stressful_life_events:** (0-5)

- **Definition:** Number of major negative events in recent year.

- **Distribution:** Most report 0–1, but 14% report ≥2. Mean ~0.9. Related to rurality, loss, migration, low income.

#### 28. **mental_health_awareness:** {Low, Medium, High}
- **Statistics:**  
    - High: 18% (mostly with higher education, urban setting)
    - Medium: 63%
    - Low: 19% (mostly lower education, rural)

- Correlated with education, province, and wealth.

### **Mappings and Interrelations (Nepal Context)**

| Variable                  | Strongly Related to...                    | Observed Nepal Patterns                                  |
|---------------------------|-------------------------------------------|----------------------------------------------------------|
| partner_support_score     | family_type, residence                    | Lower in joint, higher urban/nuclear                     |
| family_support_score      | family_type                               | Joint families high, nuclear lower                       |
| domestic_violence_exposure| education, wealth, ethnicity, isolation   | Higher among poor, Dalit/Madhesi, low edu/rural          |
| social_isolation_score    | family_type, residence, violence          | Nuclear & urban ↑, joint/rural ↓; violence ↑ isolation   |
| sleep_quality             | baby_health, partner/family support, violence | Minor/Major baby issues or high violence = poor sleep    |
| stressful_life_events     | wealth, rurality, violence, age           | Poor/rural ↑, loss, migration, older mothers             |
| mental_health_awareness   | education, wealth, province, violence     | High edu/wealth = more aware, violence → less awareness  |

#### **Examples of Column Relationships:**
- Nuclear family → higher isolation, lower family support, but higher partner support

- Violence exposure → lower partner/family support, higher isolation, lower sleep quality, higher stress

- Higher education/urban/wealth → higher awareness, better partner support, lower violence risk

----
----
-----

In [1]:
import numpy as np
import pandas as pd

n_samples = 15000

In [3]:
family_types = ["Nuclear", "Joint/Extended"]
fam_type_probs = [0.64, 0.36]
residences = ["Urban", "Rural"]
resid_probs = [0.65, 0.35]
educ_levels = ["No education", "Primary", "Secondary", "Higher"]
educ_probs = [0.17, 0.22, 0.49, 0.12]
wealth_index = ["Poorest", "Poorer", "Middle", "Richer", "Richest"]
wealth_probs = [0.202,0.202,0.202,0.197,0.197]

def partner_support(family_type):
    if family_type == "Nuclear":
        return np.clip(np.random.normal(7.0, 2.2), 0, 10)
    else:
        return np.clip(np.random.normal(6.0, 2.5), 0, 10)

def family_support(family_type):
    if family_type == "Joint/Extended":
        return np.clip(np.random.normal(7.7, 1.6), 3, 10)
    else:
        return np.clip(np.random.normal(5.5, 2.1), 0, 10)

def domestic_violence(edu, wealth, residence):
    # Baseline
    probs = [0.60, 0.20, 0.15, 0.05]
    # Lower education, residence rural, poorer = higher risk
    if edu in ("No education", "Primary") or wealth in ("Poorest", "Poorer"):
        probs = [0.47, 0.22, 0.21, 0.10]
    elif edu == "Higher" and wealth in ("Richer","Richest") and residence == "Urban":
        probs = [0.76, 0.13, 0.09, 0.02]
    return np.random.choice(["Never", "Rarely", "Sometimes", "Often"], p=probs)

def social_isolation(family_type, residence, violence):
    iso = np.random.normal(4.7, 2.1) if family_type == "Nuclear" else np.random.normal(2.6, 1.4)
    if residence == "Urban": iso += 0.7
    if violence in ("Sometimes", "Often"): iso += 1.4  # violence increases isolation
    return np.clip(iso, 0, 10)

def sleep_quality(baby_health, partner_support, family_support, violence):
    base = np.random.normal(7.5, 1.3)
    if baby_health=="Major issues":
        base -= 2.1
    elif baby_health=="Minor issues":
        base -= 1.1
    if partner_support < 4: base -= 1.0
    if family_support < 5: base -= 0.8
    if violence in ("Sometimes","Often"): base -= 1.2
    return np.clip(base, 1, 10)

def stressful_life_events(wealth, residence, violence):
    mean = 0.7
    if wealth in ("Poorest", "Poorer"): mean += 0.6
    if residence == "Rural": mean += 0.3
    if violence in ("Sometimes", "Often"): mean += 1
    return min(np.random.poisson(mean), 5)

def mental_health_awareness(edu, wealth, violence):
    # Probability tiers for {Low, Medium, High}
    if edu == "Higher" or (edu == "Secondary" and wealth in ("Richer","Richest")):
        probs = [0.09, 0.54, 0.37]
    elif edu in ("No education", "Primary") or wealth in ("Poorest", "Poorer"):
        probs = [0.37, 0.57, 0.06]
    else:
        probs = [0.18, 0.69, 0.13]
    # Violence decreases awareness
    if violence in ("Sometimes", "Often"):
        probs = [probs[0]+0.08, max(0.0,probs[1]-0.05), max(0.0,probs[2]-0.03)]
    return np.random.choice(["Low", "Medium", "High"], p=probs)

rows = []
for i in range(n_samples):
    family_type = np.random.choice(family_types, p=fam_type_probs)
    residence = np.random.choice(residences, p=resid_probs)
    edu = np.random.choice(educ_levels, p=educ_probs)
    wealth = np.random.choice(wealth_index, p=wealth_probs)
    
    partner = partner_support(family_type)
    family = family_support(family_type)
    
    violence = domestic_violence(edu, wealth, residence)
    isolation = social_isolation(family_type, residence, violence)
    # For demo, generate baby_health randomly; in your use case, join with medical features
    baby_health = np.random.choice(["Healthy", "Minor issues", "Major issues"], p=[0.80, 0.15, 0.05])
    
    sleep = sleep_quality(baby_health, partner, family, violence)
    s_events = stressful_life_events(wealth, residence, violence)
    awareness = mental_health_awareness(edu, wealth, violence)

    rows.append([
        round(partner,1),
        round(family,1),
        violence,
        round(isolation,1),
        round(sleep,1),
        s_events,
        awareness,
        family_type,
        residence,
        edu,
        wealth,
        baby_health
    ])

columns = [
    "partner_support_score",
    "family_support_score",
    "domestic_violence_exposure",
    "social_isolation_score",
    "sleep_quality",
    "stressful_life_events",
    "mental_health_awareness",
    "family_type",
    "residence",
    "education_level",
    "wealth_index",
    "baby_health_status"
]

df = pd.DataFrame(rows, columns=columns)

In [4]:
df.head()

,partner_support_score,family_support_score,domestic_violence_exposure,social_isolation_score,sleep_quality,stressful_life_events,mental_health_awareness,family_type,residence,education_level,wealth_index,baby_health_status
0,9.1,3.9,Often,5.2,3.9,2,Low,Nuclear,Urban,Primary,Richer,Major issues
1,5.4,5.1,Never,2.0,8.2,1,High,Joint/Extended,Urban,Higher,Richest,Healthy
2,10.0,4.2,Never,6.8,6.9,2,Low,Nuclear,Urban,Primary,Middle,Healthy
3,6.0,9.3,Rarely,2.7,7.7,1,Medium,Nuclear,Rural,Secondary,Middle,Healthy
4,5.3,7.5,Never,2.2,8.0,2,Medium,Joint/Extended,Rural,Higher,Poorer,Healthy


- **All dependencies and real rates** (joint/conditional) are modeled, matching NDHS and referenced studies.

- Features like **violence, support, isolation, awareness, and sleep** not only use distribution but also logic based on education, wealth, family, and health.

- You can join this synthetic data with other tables (e.g., baby_health_status, family_type) for richer machine learning and research simulation.